# Phase 4 — Leakage-Free Default Model (Application-Time Features)

**Why this notebook exists.** The Phase 3 model reports AUC ≈ 0.99 because it uses repayment-behaviour
features (`dpd_max`, `payment_ratio`, `stress_score` …). For a loan that defaulted, these are largely *the default itself*,
so they cannot be used to decide whether to approve a loan.

This notebook:
1. Demonstrates the leakage with the scored data.
2. Trains models on **only information known at origination**.
3. Evaluates on a **held-out test set split by customer** (no customer appears in both train and test).
4. Produces **out-of-fold** risk scores for every loan (each loan is scored by a model that never saw its customer).

Input: `../Data/finalscoredloans.csv`

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, brier_score_loss
from xgboost import XGBClassifier
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
os.makedirs('../Images', exist_ok=True)
SEED = 42
df = pd.read_csv('../Data/finalscoredloans.csv')
print(f'{df.shape[0]:,} loans | {df.customer_id.nunique():,} customers | default rate {df.default_flag.mean()*100:.1f}%')

6,089 loans | 4,941 customers | default rate 12.4%


## 1. Demonstrating the leakage

In [2]:
leaky = ['dpd_max','stress_score','payment_ratio','early_stress_flag']
rows=[]
for c in leaky + ['credit_bureau_score','income_to_emi','loan_amount','tenure_months']:
    a = roc_auc_score(df.default_flag, df[c]); rows.append((c, max(a,1-a)))
leak = pd.DataFrame(rows, columns=['feature','single_feature_AUC']).sort_values('single_feature_AUC',ascending=False)
leak['type'] = np.where(leak.feature.isin(leaky),'repayment (post-origination)','application-time')
print(leak.to_string(index=False))

fig, ax = plt.subplots(figsize=(7,3.6))
colors = ['#c0392b' if t.startswith('repay') else '#2e86c1' for t in leak.type]
ax.barh(leak.feature[::-1], leak.single_feature_AUC[::-1], color=colors[::-1])
ax.axvline(0.5, ls='--', c='grey', lw=1); ax.set_xlim(0.4,1.02)
ax.set_xlabel('Single-feature AUC'); ax.set_title('Repayment features (red) almost perfectly encode the outcome', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.savefig('../Images/01_leakage_single_feature_auc.png'); plt.close()

            feature  single_feature_AUC                         type
            dpd_max            0.998185 repayment (post-origination)
       stress_score            0.960704 repayment (post-origination)
      payment_ratio            0.889969 repayment (post-origination)
      tenure_months            0.737382             application-time
credit_bureau_score            0.671478             application-time
        loan_amount            0.629333             application-time
      income_to_emi            0.537876             application-time
  early_stress_flag            0.525371 repayment (post-origination)


## 2. Feature set and customer-level split

Application-time features only (known when the loan is approved). `risk_grade` is the lender's own grade at origination.
`loan_status`, `risk_category`, `segment_name`, `default_probability` are outputs of the old model/outcome and are excluded.

In [3]:
CAT = ['product_type','risk_grade','acquisition_channel']
NUM = ['loan_amount','tenure_months','income_to_emi','credit_bureau_score']
X, y, groups = df[CAT+NUM], df.default_flag.values, df.customer_id.values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr, te = next(gss.split(X, y, groups))
assert set(groups[tr]).isdisjoint(groups[te])
print(f'Train {len(tr):,} loans (default {y[tr].mean()*100:.1f}%) | Test {len(te):,} loans (default {y[te].mean()*100:.1f}%)')
print('No customer overlap between train and test: OK')

Train 4,875 loans (default 12.5%) | Test 1,214 loans (default 12.1%)
No customer overlap between train and test: OK


In [4]:
def prep(scale):
    num = StandardScaler() if scale else 'passthrough'
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CAT), ('num', num, NUM)])

def make_models(pos_w):
    return {
     'Logistic Regression': Pipeline([('p',prep(True)),('m',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=SEED))]),
     'Random Forest':       Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=30,class_weight='balanced',random_state=SEED,n_jobs=-1))]),
     'XGBoost':             Pipeline([('p',prep(False)),('m',XGBClassifier(n_estimators=250,max_depth=3,learning_rate=0.04,subsample=0.8,colsample_bytree=0.8,min_child_weight=5,reg_lambda=5,scale_pos_weight=pos_w,eval_metric='auc',random_state=SEED,verbosity=0))]),
    }
pos_w = (y[tr]==0).sum()/(y[tr]==1).sum()
models, res, probs = make_models(pos_w), [], {}
for name, pipe in models.items():
    pipe.fit(X.iloc[tr], y[tr])
    p = pipe.predict_proba(X.iloc[te])[:,1]; probs[name]=p
    res.append((name, roc_auc_score(y[te],p), average_precision_score(y[te],p)))
results = pd.DataFrame(res, columns=['model','test_AUC','test_PR_AUC']).round(4)
print(f'Baseline PR-AUC for a random model = default rate = {y[te].mean():.3f}')
print(results.to_string(index=False))

Baseline PR-AUC for a random model = default rate = 0.121
              model  test_AUC  test_PR_AUC
Logistic Regression    0.8212       0.3299
      Random Forest    0.8289       0.3431
            XGBoost    0.8297       0.3548


### Contrast: what the same XGBoost gives if repayment features are added back (leaky)

In [5]:
LEAKY = ['dpd_max','payment_ratio','stress_score','early_stress_flag']
Xl = df[CAT+NUM+LEAKY]
def prepL(): return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CAT), ('num','passthrough',NUM+LEAKY)])
leaky_model = Pipeline([('p',prepL()),('m',XGBClassifier(n_estimators=250,max_depth=3,learning_rate=0.04,scale_pos_weight=pos_w,verbosity=0,random_state=SEED))])
leaky_model.fit(Xl.iloc[tr], y[tr])
leaky_auc = roc_auc_score(y[te], leaky_model.predict_proba(Xl.iloc[te])[:,1])
print(f'XGBoost WITH repayment features (test AUC): {leaky_auc:.4f}  <- inflated, not usable for approval decisions')
print(f'XGBoost application-time only   (test AUC): {results.set_index("model").loc["XGBoost","test_AUC"]:.4f}  <- honest')

XGBoost WITH repayment features (test AUC): 1.0000  <- inflated, not usable for approval decisions
XGBoost application-time only   (test AUC): 0.8297  <- honest


## 3. ROC curves and feature importance (test set)

In [6]:
fig, axes = plt.subplots(1,2, figsize=(13,4.6))
for (name,p),col in zip(probs.items(), ['#2e86c1','#27ae60','#c0392b']):
    f,t,_ = roc_curve(y[te],p); axes[0].plot(f,t,lw=2,color=col,label=f'{name} (AUC {roc_auc_score(y[te],p):.3f})')
axes[0].plot([0,1],[0,1],'k--',lw=1,label='Random'); axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC — held-out customers', fontweight='bold'); axes[0].legend(loc='lower right', fontsize=9)

best = results.sort_values('test_AUC',ascending=False).iloc[0].model
pi = permutation_importance(models[best], X.iloc[te], y[te], scoring='roc_auc', n_repeats=20, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values()
axes[1].barh(imp.index, imp.values, xerr=pd.Series(pi.importances_std,index=X.columns)[imp.index].values, color='#2e86c1')
axes[1].set_title(f'Permutation importance — {best}\n(drop in test AUC when feature shuffled)', fontweight='bold', fontsize=10)
plt.tight_layout(); plt.savefig('../Images/02_roc_and_importance.png'); plt.close()
print('Best model by test AUC:', best); print(imp.sort_values(ascending=False).round(4).to_string())

Best model by test AUC: XGBoost
tenure_months          0.2135
risk_grade             0.0896
credit_bureau_score    0.0137
product_type           0.0050
acquisition_channel    0.0024
loan_amount           -0.0006
income_to_emi         -0.0024


## 4. Out-of-fold scores for the whole portfolio

5-fold `GroupKFold` by customer: every loan is scored by a model that never saw that customer. These scores replace the
in-sample `default_probability` for dashboards.

In [7]:
oof = np.zeros(len(df))
for k,(a,b) in enumerate(GroupKFold(n_splits=5).split(X,y,groups)):
    m = make_models((y[a]==0).sum()/(y[a]==1).sum())[best]; m.fit(X.iloc[a], y[a]); oof[b] = m.predict_proba(X.iloc[b])[:,1]
print(f'Out-of-fold AUC ({best}): {roc_auc_score(y,oof):.4f} | PR-AUC: {average_precision_score(y,oof):.4f}')
df['pd_oof'] = oof.round(4)

# Risk bands by portfolio quantiles of the score (so each band is populated and comparable)
q = np.quantile(oof,[0.50,0.80,0.95])
df['risk_band'] = pd.cut(oof, [-1,q[0],q[1],q[2],2], labels=['Low','Medium','High','Critical'])
band = df.groupby('risk_band',observed=True).agg(loans=('loan_id','count'), actual_default_rate=('default_flag','mean'), defaults=('default_flag','sum')).round(3)
band['share_of_all_defaults'] = (band.defaults/df.default_flag.sum()).round(3)
print(band.to_string())

Out-of-fold AUC (XGBoost): 0.8240 | PR-AUC: 0.3343
           loans  actual_default_rate  defaults  share_of_all_defaults
risk_band                                                             
Low         3045                0.023        69                  0.091
Medium      1826                0.145       265                  0.350
High         913                0.337       308                  0.406
Critical     305                0.380       116                  0.153


In [8]:
df['decile'] = pd.qcut(pd.Series(oof).rank(method='first'), 10, labels=range(1,11)).astype(int)
dec = df.groupby('decile').agg(loans=('loan_id','count'), default_rate=('default_flag','mean'), defaults=('default_flag','sum'))
dec = dec.sort_index(ascending=False); dec['cum_defaults_captured'] = dec.defaults.cumsum()/dec.defaults.sum()
dec['lift'] = dec.default_rate/df.default_flag.mean()
print(dec.round(3).to_string())

fig, axes = plt.subplots(1,2, figsize=(13,4.4))
b = band.reset_index()
bars = axes[0].bar(b.risk_band.astype(str), b.actual_default_rate*100, color=['#27ae60','#f1c40f','#e67e22','#c0392b'])
for r,v in zip(bars,b.actual_default_rate*100): axes[0].text(r.get_x()+r.get_width()/2, v+0.3, f'{v:.1f}%', ha='center', fontweight='bold')
axes[0].axhline(df.default_flag.mean()*100, ls='--', c='grey', lw=1, label='Portfolio avg'); axes[0].legend()
axes[0].set_ylim(0, b.actual_default_rate.max()*115); axes[0].set_ylabel('Actual default rate (%)'); axes[0].set_title('Observed default rate by out-of-fold risk band', fontweight='bold')
axes[1].plot(np.arange(0,11)*10, [0]+list(dec.cum_defaults_captured*100), marker='o', color='#2e86c1', label='Model')
axes[1].plot([0,100],[0,100],'k--',lw=1,label='Random'); axes[1].set_xlabel('% of loans reviewed (highest risk first)'); axes[1].set_ylabel('% of defaults captured')
axes[1].set_title('Cumulative gains', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.savefig('../Images/03_risk_bands_and_gains.png'); plt.close()
top20 = dec.cum_defaults_captured.iloc[1]*100
print(f'\nReviewing the riskiest 20% of loans captures {top20:.0f}% of all defaults.')

        loans  default_rate  defaults  cum_defaults_captured   lift
decile                                                             
10        609         0.378       230                  0.303  3.034
9         609         0.319       194                  0.559  2.559
8         609         0.223       136                  0.739  1.794
7         609         0.126        77                  0.840  1.016
6         608         0.086        52                  0.909  0.687
5         609         0.077        47                  0.971  0.620
4         609         0.028        17                  0.993  0.224
3         609         0.008         5                  1.000  0.066
2         609         0.000         0                  1.000  0.000
1         609         0.000         0                  1.000  0.000



Reviewing the riskiest 20% of loans captures 56% of all defaults.


## 5. Save outputs

In [9]:
out = df[['loan_id','customer_id','product_type','risk_grade','loan_amount','tenure_months','acquisition_channel',
          'default_flag','loan_status','credit_bureau_score','income_to_emi','pd_oof','risk_band']]
out.to_csv('../Data/scored_loans_leakage_free.csv', index=False)
results.to_csv('../Data/model_comparison.csv', index=False)
print('Saved Data/scored_loans_leakage_free.csv and Data/model_comparison.csv')
print(results.to_string(index=False))

Saved Data/scored_loans_leakage_free.csv and Data/model_comparison.csv
              model  test_AUC  test_PR_AUC
Logistic Regression    0.8212       0.3299
      Random Forest    0.8289       0.3431
            XGBoost    0.8297       0.3548
